In [ ]:
!pip install -q sentence-transformers #Creates Embeddings
!pip install -q faiss-cpu #Vector DB
!pip install -q transformers #Load LLM
!pip install -q accelerate
!pip install -q pypdf #Read pdfs

In [ ]:
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import numpy as np
import faiss

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [ ]:
pdf_files = [
    "DL research.pdf",
    "SQL.pdf",
    "EDA on SQL .pdf",
    "sql interview question.pdf",
    "PBI questions.pdf",
    "stats..pdf"
]

text = ""

for file in pdf_files:

    reader = PdfReader(file)

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text

In [ ]:
chunk_size = 500
chunks = []

for i in range(0, len(text), chunk_size):

    chunks.append(text[i:i+chunk_size])

print("Total chunks:", len(chunks))

Total chunks: 1020


In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(chunks)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Vector database ready")

Vector database ready


# LLM

## TinyLlama

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name_tiny = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer_tiny = AutoTokenizer.from_pretrained(model_name_tiny)

model_tiny = AutoModelForCausalLM.from_pretrained(
    model_name_tiny,
    device_map="auto",
    torch_dtype="auto"
)

generator_tiny = pipeline(
    "text-generation",
    model=model_tiny,
    tokenizer=tokenizer_tiny
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## Phi 2

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, pipeline

model_name = "microsoft/phi-2"

config = AutoConfig.from_pretrained(model_name)

config.pad_token_id = config.eos_token_id

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,
    device_map="auto",
    torch_dtype="auto"
)

generator_phi = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    max_length = None
)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

In [ ]:
def retrieve_context(question):

    question_embedding = embedding_model.encode([question])

    distances, indices = index.search(
        np.array(question_embedding), k=3
    )

    results = [chunks[i] for i in indices[0]]

    return " ".join(results)

In [ ]:
def ask_tiny(question):

    context = retrieve_context(question)

    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant. Answer the question briefly in 2-3 sentences."
        },
        {
            "role": "user",
            "content": f"Question: {question}\n\nContext: {context}"
        }
    ]

    prompt = tokenizer_tiny.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    result = generator_tiny(
        prompt,
        max_new_tokens=80,
        do_sample=False,
        return_full_text=False
    )

    return result[0]["generated_text"].strip()

In [ ]:
def ask_phi(question):

    context = retrieve_context(question)

    prompt = f"""
Answer the question in 3 sentences.

Question: {question}

Context: {context}

Answer:
"""

    result = generator_phi(
        prompt,
        max_new_tokens=120,
        do_sample=False,
        return_full_text=False
    )

    return result[0]["generated_text"]

In [ ]:
def score_models(phi_answer, tiny_answer, context):

    phi_vec = embedding_model.encode(phi_answer)
    tiny_vec = embedding_model.encode(tiny_answer)
    context_vec = embedding_model.encode(context)

    # cosine similarity
    phi_sim = np.dot(phi_vec, context_vec) / (
        np.linalg.norm(phi_vec) * np.linalg.norm(context_vec)
    )

    tiny_sim = np.dot(tiny_vec, context_vec) / (
        np.linalg.norm(tiny_vec) * np.linalg.norm(context_vec)
    )

    # normalize to 0–1
    phi_score = (phi_sim + 1) / 2
    tiny_score = (tiny_sim + 1) / 2

    return phi_score, tiny_score

In [ ]:
def compare_models(question):

    # Get answers from both models
    phi_answer = ask_phi(question)
    tiny_answer = ask_tiny(question)

    # Retrieve context
    contexts = retrieve_context(question)

    # Score answers against context
    phi_score, tiny_score = score_models(phi_answer, tiny_answer, contexts[0])

    # Decide final answer
    if phi_score > tiny_score:
        final_answer = phi_answer
        winner = "Phi-2"
    else:
        final_answer = tiny_answer
        winner = "TinyLlama"

    print("\nQuestion:")
    print(question)

    print("\nPhi-2 Answer:")
    print(phi_answer)

    print("\nTinyLlama Answer:")
    print(tiny_answer)

    print("\nFinal Answer Selected:")
    print(final_answer)

    print("\nWinning Model:", winner)

    print("\nModel Scores (0–1 scale)")
    print("Phi-2 Score:", round(phi_score,2))
    print("TinyLlama Score:", round(tiny_score,2))

In [ ]:
compare_models("What is statistics?")


Question:
What is statistics?

Phi-2 Answer:

Statistics is the science of collecting, organizing, and analyzing data. It is used to make conclusions about the population by using analytical tools on the sample. Descriptive statistics are used to organize and summarize data, while inferential statistics are used to make conclusions about the population. Measures of central tendency, such as mean, median, and mode, are used to describe the center of a dataset. Measures of dispersion, such as range, variance, and standard deviation, are used to describe the spread of a dataset. Different types of distributions, such as Bernoulli, uniform, binomial

TinyLlama Answer:
Statistics is the science of collecting, organizing, and analyzing data to gain insights and make informed decisions. Data is defined as facts or pieces of information, and statistics are the science of collecting, organizing, and analyzing data to provide insights and make decisions. Data is collected from various sources, 

In [ ]:
def chatbot():

    print("RAG Chatbot Started")
    print("Type 'exit' to stop\n")

    while True:

        question = input("\nAsk a question: ")

        if question.lower() == "exit":
            print("Chat ended.")
            break

        print("\nChoose model:")
        print("1 - Phi-2")
        print("2 - TinyLlama")
        print("3 - Compare both")

        choice = input("Enter choice: ")

        if choice == "1":

            answer = ask_phi(question)

            print("\nPhi-2 Answer:")
            print(answer)

        elif choice == "2":

            answer = ask_tiny(question)

            print("\nTinyLlama Answer:")
            print(answer)

        elif choice == "3":

            compare_models(question)

        else:

            print("Invalid option. Try again.")


# START CHATBOT
chatbot()